# Portfolio Competition — Teacher Control Panel

**Private notebook · Google Colab · 90-minute competition**

This notebook manages the full competition: it downloads the official prices, reads team submissions from a **private** Google Sheet, validates portfolios, enforces the **four-attempt limit**, computes the private Score, and publishes **only** the five leaderboard columns to the public Google Sheet.

**Run sections 1–4 once in order.** During the competition, run only `refresh_and_publish()` to process new Google Form submissions. Do not share this notebook with students or project its output containing private data.

**Privacy:** `Display` is the public presentation tab and is never edited by this notebook. The only written cells are `Leaderboard!A1:E16`.

## 1. Configuration

All competition settings and URLs are in one place. Change the URLs only if the Google Sheets files are replaced.

**Scoring rule (private):** lower is better; annualized volatility is in decimal units.

In [1]:
import re
import numpy as np
import pandas as pd
from IPython.display import display

# ----- Competition rules (PRIVATE) -----
MIN_K = 5
MAX_K = 30
MAX_ATTEMPTS = 4
NUM_TEAMS = 15
PENALTY = 0.05  # Score points per stock above MIN_K
TRADING_DAYS = 252

# Data rules
MAX_MISSING = 0.20
EXPECTED_STOCKS = 325
EXPECTED_RETURN_DATES = 6832

DATA_URL = (
    "https://raw.githubusercontent.com/alfonso-santos/"
    "microcredencial-carteras-python/Finance_Technology/"
    "Tema_2_Carteras_Intro/data/SP500_comp_data.xlsx"
)

# Google Sheets. The first sheet is PRIVATE; never give students its URL.
PRIVATE_SHEET_URL = (
    "https://docs.google.com/spreadsheets/d/"
    "19wEbQioeZbN8vz-bp3i_XChFeC4QybdGLPu0xobHXFc/edit"
)
PUBLIC_SHEET_URL = (
    "https://docs.google.com/spreadsheets/d/"
    "1rPJvoMuA0R9UQM_pbD18fcA6ZismvDRQUskG2Y_oAxI/edit"
)

TEAM_NAMES = {f"Team {i}" for i in range(1, NUM_TEAMS + 1)}
PUBLIC_COLUMNS = ["Rank", "Team", "Performance", "Attempts", "Change"]

assert 1 <= MIN_K <= MAX_K
assert MAX_ATTEMPTS > 0 and NUM_TEAMS > 0
assert PENALTY >= 0 and 0 <= MAX_MISSING < 1

## 2. Download and clean the official data

- Convert invalid (non-numeric, zero, negative, non-finite) prices to missing values.
- Exclude stocks with **more than 20%** missing or invalid price observations.
- Remove dates with any remaining missing prices. **No imputation.**
- Calculate daily log returns and check that the official sample has not changed.

If the number of stocks or dates differs from the reference, the code **stops** rather than silently grading portfolios with a different sample.

In [2]:
# Read and standardize the official price file.
prices = pd.read_excel(DATA_URL, index_col="Date")
prices.index = pd.to_datetime(prices.index, errors="raise")
prices = prices.sort_index()
prices.columns = [str(t).strip().upper() for t in prices.columns]

if prices.index.has_duplicates or prices.columns.duplicated().any():
    raise ValueError("Duplicate dates or ticker names in the price file.")

original_shape = prices.shape
prices = prices.apply(pd.to_numeric, errors="coerce")
prices = prices.where(np.isfinite(prices) & (prices > 0))

# Keep stocks with at least 80% valid prices.
prices = prices.loc[:, prices.isna().mean() <= MAX_MISSING]

# Keep common complete price dates; then compute log returns.
prices = prices.dropna()
returns = np.log(prices / prices.shift(1)).dropna()

print(f"Original prices: {original_shape[1]} stocks × {original_shape[0]} dates")
print(f"Cleaned returns: {returns.shape[1]} stocks × {len(returns)} dates")
print(f"Period: {returns.index.min().date()} to {returns.index.max().date()}")

if (returns.shape[1], len(returns)) != (EXPECTED_STOCKS, EXPECTED_RETURN_DATES):
    raise RuntimeError(
        "OFFICIAL SAMPLE MISMATCH — stop the competition. "
        f"Expected {EXPECTED_STOCKS} stocks and {EXPECTED_RETURN_DATES} return dates."
    )
if not np.isfinite(returns.to_numpy()).all():
    raise RuntimeError("Official returns contain missing or non-finite values.")

Original prices: 504 stocks × 8475 dates
Cleaned returns: 325 stocks × 6832 dates
Period: 1996-07-01 to 2023-08-22


## 3. Authenticate with Google and connect both Sheets

Use the **same Google account** that owns the private submissions sheet and the public leaderboard. Colab will request authorization. Students must have **read-only** access to the public spreadsheet, and **no access** to the private one.

In [3]:
import gspread
from google.colab import auth
from google.auth import default

auth.authenticate_user()
credentials, _ = default()
gc = gspread.authorize(credentials)

# Private Google Form responses (read only)
private_book = gc.open_by_url(PRIVATE_SHEET_URL)
# Portfolio submissions from Google Forms
submissions_sheet = private_book.worksheet("Portfolio_Submissions")

# Public raw ranking (write only its A:E content)
public_book = gc.open_by_url(PUBLIC_SHEET_URL)
public_sheet = public_book.worksheet("Leaderboard")
public_book.worksheet("Display")  # Confirm the separate presentation tab exists.

# Verify the form's column names before proceeding.
form_headers = submissions_sheet.row_values(1)
required = {"Team name", "Selected stock tickers"}
if not required.issubset(form_headers):
    raise ValueError(f"Unexpected form columns: {form_headers}")

print("Connected to private submissions and public leaderboard.")

Connected to private submissions and public leaderboard.


### GENERA LOS CÓDIGOS DE CADA EQUIPO

Los códigos se guardan en la google sheet Portfolio Challenge - Private Submissions y hay que dárselo a cada equipo

### GENERA LOS CODIGOS LA PRIMERA VEZ (DESCOMENTAR PARA GENERAR)

In [25]:

# import secrets
# from datetime import datetime

# # Private worksheets
# codes_sheet = private_book.worksheet("Team_Codes")
# registry_sheet = private_book.worksheet("Team_Registry")

# # Official teams
# teams = [f"Team {i}" for i in range(1, NUM_TEAMS + 1)]

# # Check that both worksheets contain the expected teams
# assert codes_sheet.col_values(1)[1:] == teams
# assert registry_sheet.col_values(1)[1:] == teams

# # Reserve column C to remember that codes have been renewed
# if codes_sheet.col_count < 3:
#     codes_sheet.add_cols(3 - codes_sheet.col_count)

# if not codes_sheet.acell("C2").value:

#     # Generate new unique codes
#     alphabet = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
#     old_codes = set(codes_sheet.col_values(2)[1:])
#     new_codes = {}

#     for team in teams:
#         while True:
#             code = "".join(secrets.choice(alphabet) for _ in range(8))
#             if code not in old_codes and code not in new_codes.values():
#                 break
#         new_codes[team] = code

#     # Update Team_Codes
#     codes_sheet.update(
#         range_name=f"B2:B{NUM_TEAMS + 1}",
#         values=[[new_codes[t]] for t in teams]
#     )

#     # Update Team_Registry with exactly the same codes
#     registry_sheet.update(
#         range_name=f"B2:B{NUM_TEAMS + 1}",
#         values=[[new_codes[t]] for t in teams]
#     )

#     # Mark the rotation as completed
#     codes_sheet.update(
#         range_name="C1:C2",
#         values=[["Last code rotation"], [datetime.now().isoformat()]]
#     )

#     print("New team codes generated and saved.")

# else:
#     print("Codes have already been renewed. Existing codes preserved.")

# # Reload the official codes into Colab
# team_codes = {
#     row["Team"]: str(row["Code"])
#     for row in codes_sheet.get_all_records()
# }

# # Verify that both worksheets contain matching codes
# registry_codes = {
#     row["Team"]: str(row["Secret code"])
#     for row in registry_sheet.get_all_records()
# }

# assert set(team_codes) == set(teams)
# assert team_codes == registry_codes
# assert len(set(team_codes.values())) == NUM_TEAMS

# print(f"Validation passed: {NUM_TEAMS} matching team codes.")


New team codes generated and saved.
Validation passed: 15 matching team codes.


### ESTA FUNCION SOLO LEE LOS CODIGOS QUE YA EXISTEN Y, SI YA EXISTEN, LOS USA

In [4]:

import secrets


# Teams and allowed characters
teams = [f"Team {i}" for i in range(1, NUM_TEAMS + 1)]
alphabet = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"

# Open or create a private worksheet for team codes
try:
    codes_sheet = private_book.worksheet("Team_Codes")
except gspread.WorksheetNotFound:
    codes_sheet = private_book.add_worksheet(
        title="Team_Codes", rows=20, cols=2
    )

# Read existing codes, if any
existing = codes_sheet.get_all_records()

if existing:
    # Reuse codes from a previous session
    team_codes = {
        row["Team"]: str(row["Code"])
        for row in existing
    }
    print("Existing team codes loaded.")

else:
    # Generate one unique random code per team
    team_codes = {}

    for team in teams:
        while True:
            code = "".join(secrets.choice(alphabet) for _ in range(8))
            if code not in team_codes.values():
                break

        team_codes[team] = code

    # Save the codes in the private Google Sheet
    data = [["Team", "Code"]] + list(team_codes.items())
    codes_sheet.update(range_name="A1", values=data)

    print("15 team codes created and saved.")

# Check that all teams have a different code
assert set(team_codes) == set(teams)
assert len(set(team_codes.values())) == NUM_TEAMS

print("Team code validation passed.")


Existing team codes loaded.
Team code validation passed.


## 4. Portfolio validation and scoring

We accept straightforward copy-and-paste errors: extra spaces, lower-case letters, and single/double quotation marks around tickers. Examples such as `AAPL, 'MSFT', "JNJ"` or `['AAPL', 'MSFT', 'JNJ']` are parsed correctly.

The function rejects: incorrect portfolio size, repeated tickers, names not in the **cleaned** investment universe, malformed lists, and non-finite results. Invalid *new* submissions still use an official attempt. An exact duplicate of a previous portfolio from the same team does **not** use another attempt.

In [5]:
def parse_tickers(text, available=None):
    """Read comma-separated tickers, tolerating surrounding spaces and quotes."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Empty ticker list.")

    raw = text.strip().replace("“", '"').replace("”", '"').replace("’", "'")
    # Also accept a pasted Python list: ['AAPL', 'MSFT', ...]
    if len(raw) >= 2 and (raw[0], raw[-1]) in {("[", "]"), ("(", ")")}:
        raw = raw[1:-1].strip()

    parts = re.split(r"[,;\n]", raw)
    tickers = [p.strip().strip("'\"").strip().upper() for p in parts]
    if not tickers or any(not t for t in tickers):
        raise ValueError("Blank ticker detected. Separate tickers with commas.")

    if available is not None:
        universe = set(available)
        # Resolve known dot/dash class share spelling only when unambiguous.
        fixed = []
        for ticker in tickers:
            if ticker in universe:
                fixed.append(ticker)
            elif ticker.replace(".", "-") in universe:
                fixed.append(ticker.replace(".", "-"))
            elif ticker.replace("-", ".") in universe:
                fixed.append(ticker.replace("-", "."))
            else:
                fixed.append(ticker)
        tickers = fixed

    return tickers


def normalize_team(name):
    """Convert ' team 01 ' to 'Team 1'; reject unknown team numbers."""
    match = re.fullmatch(r"team\s*0*(\d+)", str(name).strip(), re.IGNORECASE)
    if not match:
        return None
    team = f"Team {int(match.group(1))}"
    return team if team in TEAM_NAMES else None


def evaluate_portfolio(tickers_text, return_data=None):
    """Validate one submitted portfolio and calculate the PRIVATE Score."""
    data = returns if return_data is None else return_data
    tickers = parse_tickers(tickers_text, data.columns)
    k = len(tickers)

    if not MIN_K <= k <= MAX_K:
        raise ValueError(f"Expected {MIN_K}–{MAX_K} tickers; received {k}.")
    if len(set(tickers)) != k:
        raise ValueError("The same stock appears more than once.")

    unknown = sorted(set(tickers) - set(data.columns))
    if unknown:
        raise ValueError(f"Tickers outside the eligible universe: {', '.join(unknown)}")

    # All assets share the same return dates; equal weights are 1/K.
    daily_portfolio = data[tickers].mean(axis=1)
    volatility = float(daily_portfolio.std(ddof=1) * np.sqrt(TRADING_DAYS))
    if not np.isfinite(volatility):
        raise ValueError("Unable to compute finite annualized volatility.")

    score = 100 * volatility + PENALTY * (k - MIN_K)
    return {
        "Tickers": ",".join(sorted(tickers)),
        "Stocks": k,
        "Volatility": volatility,
        "Score": score,
    }

## 5. Process all submissions safely

Each refresh **reconstructs** the competition from Google Forms rows in order. This prevents double-counting when a cell is run repeatedly or Colab restarts.

Rules:

- A new, different portfolio consumes **one of four attempts**, including an invalid portfolio.
- A duplicate portfolio submitted again by the **same team** does not consume another attempt, regardless of ticker order, spaces or quotes.
- Submissions after attempt 4 are ignored.
- Only the team's best **valid** Score enters the leaderboard.
- An unrecognized team name is recorded in the private audit but cannot be assigned an attempt.

The detailed audit and the best portfolios remain **private in Colab**.

In [23]:
def submission_key(raw_text, available):
    """Create a stable key for deduplication, including malformed entries."""
    try:
        return ("TICKERS", tuple(sorted(parse_tickers(raw_text, available))))
    except ValueError:
        return ("MALFORMED", re.sub(r"\s+", "", str(raw_text)).upper())


def process_submissions(rows, previous_ranks=None, return_data=None):
    """Validate team codes and portfolios, then calculate the leaderboard."""
    data = returns if return_data is None else return_data
    previous_ranks = {} if previous_ranks is None else previous_ranks
    attempts, seen, best, history, best_row = {}, {}, {}, [], {}


    for row_number, row in enumerate(rows, start=2):

        # Ignore completely empty rows in Google Sheets
        if not any(str(value).strip() for value in row.values()):
            continue

        original_team = str(row.get("Team name", "")).strip()
        team = normalize_team(original_team)
        text = str(row.get("Selected stock tickers", "")).strip()



        # Check that the team exists
        if team is None:
            history.append({
                "Row": row_number,
                "Team": original_team,
                "Attempt": 0,
                "Status": "Unknown team",
                "Detail": "Invalid team name."
            })
            continue

        attempts.setdefault(team, 0)

        # Validate the team's secret code
        submitted_code = str(row.get("Team Secret Code", "")).strip().upper()
        expected_code = team_codes.get(team, "").strip().upper()

        if not expected_code or not secrets.compare_digest(
            submitted_code, expected_code
        ):
            history.append({
                "Row": row_number,
                "Team": team,
                "Attempt": attempts[team],
                "Status": "Invalid team code — not counted",
                "Detail": "Incorrect or missing team code."
            })
            continue

        seen.setdefault(team, set())

        key = submission_key(text, data.columns)
        details = ""

        if key in seen[team]:
            status = "Duplicate — not counted"
        elif attempts[team] >= MAX_ATTEMPTS:
            status = "Attempt limit reached"
        else:
            seen[team].add(key)

            try:
                # Validate and evaluate the portfolio
                result = evaluate_portfolio(text, data)

                # Only valid portfolios consume an attempt
                attempts[team] += 1
                status = "Accepted"

                # Keep the team's best portfolio
                if team not in best or result["Score"] < best[team]["Score"]:
                  best[team] = result
                  best_row[team] = row_number

            except ValueError as exc:
                # Invalid portfolios do not consume an attempt
                status = "Invalid — not counted"
                details = str(exc)

        history.append({"Row": row_number, "Team": team,
                        "Attempt": attempts[team], "Status": status,
                        "Detail": details})

    audit = pd.DataFrame(history, columns=["Row", "Team", "Attempt", "Status", "Detail"])

    if not best:
        return pd.DataFrame(columns=PUBLIC_COLUMNS), audit, pd.DataFrame(), attempts

    # Never expose the next table or the Score publicly.

    # Private results: never publish tickers or Score
    private_best = pd.DataFrame([
        {
            "Team": team,
            **result,
            "Attempts": attempts[team],
            "Submission order": best_row[team]
        }
        for team, result in best.items()
    ])

    # Sort by Score, volatility, number of stocks and submission order
    private_best = private_best.sort_values(
        ["Score", "Volatility", "Stocks", "Submission order"],
        kind="stable"
    ).reset_index(drop=True)

    # Assign a unique rank to each team
    private_best["Rank"] = np.arange(1, len(private_best) + 1)

    leader = float(private_best["Score"].min())


    public = private_best[["Rank", "Team"]].copy()
    if leader == 0:
        public["Performance"] = np.where(private_best["Score"] == 0, 100.0, 0.0)
    else:
        public["Performance"] = 100 * leader / private_best["Score"]
    public["Performance"] = public["Performance"].round(2)
    public["Attempts"] = private_best["Attempts"].astype(str) + f"/{MAX_ATTEMPTS}"

    def change(team, rank):
        if team not in previous_ranks:
            return "NEW"
        old = previous_ranks[team]
        return f"↑ {old-rank}" if rank < old else f"↓ {rank-old}" if rank > old else "—"

    public["Change"] = [change(t, int(r)) for t, r in zip(public["Team"], public["Rank"])]
    return public[PUBLIC_COLUMNS], audit, private_best, attempts


def get_published_ranks():
    """Read ONLY prior public team/rank values to show ranking changes."""
    rows = public_sheet.get_all_records()
    previous = {}
    for row in rows:
        try:
            previous[str(row["Team"]).strip()] = int(row["Rank"])
        except (KeyError, ValueError, TypeError):
            pass
    return previous

## 6. One-command refresh: read → grade → publish

**During the competition:** run `refresh_and_publish()` whenever you want to update the public leaderboard.

The publication updates **only** `Leaderboard!A1:E16`, preserving the public `Display` tab, medals, formulas and formatting. It clears any old team rows when the form contains fewer teams or is empty. Repeated executions with no new submissions simply show the current ranking.

Private details are accessible as `attempt_history` and `private_best`, but these are **never** written to the public spreadsheet.

In [7]:
# Cache for this Colab session: do not replace the public change indicator
# with a dash merely because the teacher presses Refresh twice.
last_submission_signature = None
last_public_table = None


def refresh_and_publish():
    global last_submission_signature, last_public_table
    global attempt_history, private_best, leaderboard, attempts_by_team

    # Read all Form responses, including the test rows currently in the sheet.
    rows = submissions_sheet.get_all_records()
    signature = tuple((str(r.get("Team name", "")),
                       str(r.get("Selected stock tickers", "")),
                       str(next(iter(r.values()), ""))) for r in rows)

    if signature == last_submission_signature and last_public_table is not None:
        print("No new submissions. Leaderboard unchanged.")
        display(last_public_table)
        return last_public_table

    # Changes are measured against the previous PUBLICATION, not old Colab memory.
    previous_ranks = get_published_ranks()
    leaderboard, attempt_history, private_best, attempts_by_team = process_submissions(
        rows, previous_ranks=previous_ranks
    )

    # One Sheets update. Always write a header plus 15 team rows,
    # padding unused rows with empty cells to remove stale names.
    published_rows = []
    for _, row in leaderboard.iterrows():
        published_rows.append([
            int(row["Rank"]), row["Team"], f"{row['Performance']:.2f}",
            row["Attempts"], row["Change"]
        ])
    payload = [PUBLIC_COLUMNS] + published_rows
    payload += [[""] * 5 for _ in range(NUM_TEAMS - len(published_rows))]

    # This sheet is public. Absolutely no tickers, volatility or private Score.
    public_sheet.update(range_name=f"A1:E{NUM_TEAMS+1}", values=payload,
                        value_input_option="RAW")

    last_submission_signature = signature
    last_public_table = leaderboard.copy()
    print(f"Received: {len(rows)} submissions | Ranked: {len(leaderboard)} teams")
    display(leaderboard)
    return leaderboard

## 7. Offline checks — run before the competition

These tests use **synthetic daily returns**, do not access Google Sheets and **do not consume real attempts**. They cover spaces and quotes, invalid tickers, repeated assets, duplicate submissions, attempt 5, ranking order, and empty data.

Run the cell once. It must print `All offline checks passed.`

In [10]:
# A small, self-contained synthetic return panel (no internet required).
rng = np.random.default_rng(7)
test_symbols = ["AAPL", "ABC", "ABT", "ACGL", "ADBE", "ADI", "AMZN", "JNJ"]
test_returns = pd.DataFrame(
    rng.normal(0, 0.01, size=(80, len(test_symbols))), columns=test_symbols
)

# Quotes, spaces, brackets and lowercase are accepted.
a = evaluate_portfolio(" 'aapl', \"abc\", 'ABT', ACGL, ADBE ", test_returns)
b = evaluate_portfolio("['AAPL','ABC','ABT','ACGL','ADBE']", test_returns)
assert a["Tickers"] == b["Tickers"]
assert np.isclose(a["Volatility"], b["Volatility"])

for invalid in ["AAPL,ABC,ABT,ACGL", "AAPL,ABC,ABT,ACGL,AAPL",
                "AAPL,ABC,ABT,ACGL,FAKE"]:
    try:
        evaluate_portfolio(invalid, test_returns)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected a validation error: {invalid}")



trial_rows = [
    # First valid submission
    {
        "Team name": "Team 1",
        "Team secret code": team_codes["Team 1"],
        "Selected stock tickers": "AAPL,ABC,ABT,ACGL,ADBE"
    },

    # Duplicate: does not count
    {
        "Team name": "Team 1",
        "Team secret code": team_codes["Team 1"],
        "Selected stock tickers": "'ADBE', 'ABT', 'ABC', 'AAPL', 'ACGL'"
    },

    # Invalid ticker: does not count
    {
        "Team name": "Team 1",
        "Team secret code": team_codes["Team 1"],
        "Selected stock tickers": "AAPL,ABC,ABT,ACGL,FAKE"
    },

    # Second, third and fourth valid submissions
    {
        "Team name": "Team 1",
        "Team secret code": team_codes["Team 1"],
        "Selected stock tickers": "AAPL,ABC,ABT,ACGL,ADI"
    },
    {
        "Team name": "Team 1",
        "Team secret code": team_codes["Team 1"],
        "Selected stock tickers": "AAPL,ABC,ABT,ACGL,AMZN"
    },
    {
        "Team name": "Team 1",
        "Team secret code": team_codes["Team 1"],
        "Selected stock tickers": "AAPL,ABC,ABT,ACGL,JNJ"
    },

    # Fifth valid portfolio: attempt limit reached
    {
        "Team name": "Team 1",
        "Team secret code": team_codes["Team 1"],
        "Selected stock tickers": "AAPL,ABC,ABT,ADBE,JNJ"
    },

    # Incorrect secret code: does not count
    {
        "Team name": "Team 2",
        "Team secret code": "WRONGCODE",
        "Selected stock tickers": "AAPL,ABC,ABT,ACGL,ADBE"
    },

    # Correct code: Team 2's first valid attempt
    {
        "Team name": "Team 2",
        "Team secret code": team_codes["Team 2"],
        "Selected stock tickers": "AAPL,ABC,ABT,ACGL,ADI"
    },

    # Unknown team: rejected
    {
        "Team name": "Team 16",
        "Team secret code": "INVALID",
        "Selected stock tickers": "AAPL,ABC,ABT,ACGL,ADI"
    }
]


# Use the same column name as the real Google Form
for row in trial_rows:
    row["Team Secret Code"] = row.pop("Team secret code")




# Process synthetic submissions

board, log, winners, used = process_submissions(
    trial_rows, return_data=test_returns
)

# Check attempt limits
assert used["Team 1"] == 4
assert used["Team 2"] == 1

# Check submission validation
assert list(log["Status"]).count("Duplicate — not counted") == 1
assert list(log["Status"]).count("Invalid — not counted") == 1
assert list(log["Status"]).count("Invalid team code — not counted") == 1
assert list(log["Status"]).count("Attempt limit reached") == 1
assert list(log["Status"]).count("Unknown team") == 1

# Check leaderboard
assert len(board) == 2
assert board["Performance"].max() == 100.0

# Empty submissions produce an empty leaderboard
assert len(process_submissions([], return_data=test_returns)[0]) == 0

print("All offline checks passed.")



All offline checks passed.


## 8. Operational notes for the 90-minute session

**Before class:**

1. Perform the offline checks above. Confirm the private Form response sheet and the public leaderboard are connected.
2. **Delete test submissions from both places:** Google Forms **Responses**, and the corresponding rows in the **linked private Google Sheets**. Deleting responses in Forms alone may leave historical rows in Sheets. **Do not delete the header row.**
3. Run `refresh_and_publish()`. Check the public `Leaderboard` table is empty except for the headers, and `Display` shows no team names. If you have just erased all rows, the empty leaderboard will be published automatically.
4. Give students only the **Forms URL** and the **public Display/leaderboard URL**. Never share this notebook or the private sheet.

**During class:**

- Teams submit via Google Forms. Run `refresh_and_publish()` whenever you want to update the ranking.
- You may inspect `attempt_history` and `private_best` **privately**; never project them.
- Attempt 4 is the last counted submission. Duplicate combinations (same stocks, different order) do not consume another attempt.
- Do not change the reference dataset, the parameters, or the source Sheet while teams are competing.

**After class:** export the private results if desired and reveal the Score formula. The public `Display` is never modified by the grading code.

### Private audit (optional; do not project)

To review rejected attempts or best portfolios, run the following **only on the teacher's private screen**. The public Display will remain unchanged.

In [ ]:
# Uncomment only when you need to inspect errors privately.
# display(attempt_history)
# display(private_best)

## COMPROBACIÓN DE LOS EQUIPOS REGISTRADOS


In [14]:

from collections import defaultdict
import re

def check_team_registrations():

    # Read the private team registry and secret codes
    registry = pd.DataFrame(
        private_book.worksheet("Team_Registry").get_all_records()
    )

    codes = pd.DataFrame(
        private_book.worksheet("Team_Codes").get_all_records()
    )

    # Official list of teams and codes
    teams = [f"Team {i}" for i in range(1, NUM_TEAMS + 1)]
    official_codes = dict(zip(codes["Team"], codes["Code"]))

    issues = []
    email_teams = defaultdict(set)

    # Check every official team
    for team in teams:

        records = registry[registry["Team"] == team]

        if records.empty:
            issues.append([team, "Missing registration row", ""])
            continue

        row = records.iloc[0]

        # Number of registration submissions
        count = int(row["Registration submissions"] or 0)
        status = str(row["Status"]).strip()

        if count == 0:
            issues.append([team, "Not registered", ""])

        if count > 1:
            issues.append([
                team, "Duplicate registration",
                f"{count} registrations received"
            ])

        if count > 0 and status != "REGISTERED":
            issues.append([team, "Registration needs review", status])

        # Read registered email addresses
        emails = [
            str(row[f"Member {i} email"]).strip().lower()
            for i in range(1, 5)
        ]

        emails = [email for email in emails if email]

        # Check the number of members
        if count > 0 and not 2 <= len(emails) <= 4:
            issues.append([
                team, "Invalid number of members",
                f"{len(emails)} members"
            ])

        # Detect repeated emails within the team
        if len(emails) != len(set(emails)):
            issues.append([team, "Duplicate email within team", ""])

        # Check email format and collect emails for cross-team checks
        for email in emails:

            if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email):
                issues.append([team, "Invalid email format", email])

            email_teams[email].add(team)

        # Verify that the secret codes match
        registry_code = str(row["Secret code"]).strip()
        official_code = str(official_codes.get(team, "")).strip()

        if not official_code or registry_code != official_code:
            issues.append([
                team, "Secret code mismatch",
                "Check Team_Codes and Team_Registry"
            ])

    # Detect students registered in multiple teams
    for email, assigned_teams in email_teams.items():

        if len(assigned_teams) > 1:
            issues.append([
                ", ".join(sorted(assigned_teams)),
                "Email registered in multiple teams",
                email
            ])

    # Create the diagnostic report
    report = pd.DataFrame(
        issues, columns=["Team", "Issue", "Details"]
    )

    registered = sum(
        1 for team in teams
        if not report["Team"].eq(team).any()
    ) if not report.empty else len(teams)

    print(f"Total teams: {len(teams)}")
    print(f"Teams without detected issues: {registered}")
    print(f"Issues detected: {len(report)}")

    if report.empty:
        print("All team registrations are valid.")
    else:
        display(report)

    return report



In [15]:
# EJECUTA LA COMPROBACIÓN

# Check team registrations
registration_issues = check_team_registrations()


Total teams: 15
Teams without detected issues: 3
Issues detected: 12


,Team,Issue,Details
0,Team 2,Not registered,
1,Team 4,Not registered,
2,Team 5,Registration needs review,REVIEW — DUPLICATE EMAILS
3,Team 6,Not registered,
4,Team 7,Not registered,
5,Team 8,Not registered,
6,Team 9,Not registered,
7,Team 11,Not registered,
8,Team 12,Not registered,
9,Team 13,Not registered,


## COMPROBAR SUBMISIONES CON ERRORES


In [20]:

def check_portfolio_submissions(team=None):
    """Check portfolio submissions and display validation problems."""

    # Read the latest submissions from Google Sheets
    rows = submissions_sheet.get_all_records()

    if not rows:
        print("No portfolio submissions received.")
        return pd.DataFrame()

    # Check that Google Sheets contains the expected columns
    required = {
        "Team name",
        "Team Secret Code",
        "Selected stock tickers"
    }

    columns = set(submissions_sheet.row_values(1))
    missing = required - columns

    if missing:
        print(f"Missing Google Sheets columns: {sorted(missing)}")
        return pd.DataFrame()

    # Recalculate all submissions without modifying Google Sheets
    _, audit, _, attempts = process_submissions(rows)

    valid_rows = [
    row for row in rows
    if any(str(value).strip() for value in row.values())
]

    print(f"Total submissions: {len(valid_rows)}")
    print(f"Accepted submissions: {(audit['Status'] == 'Accepted').sum()}")
    print(f"Submissions with problems: {(audit['Status'] != 'Accepted').sum()}")

    if team is not None:
        # Show all submissions for a specific team
        team = str(team).strip()

        report = audit[
            audit["Team"].str.casefold() == team.casefold()
        ]

        print(f"\n{team}: {attempts.get(team, 0)}/{MAX_ATTEMPTS} attempts used")

    else:
        # Show only problematic submissions across all teams
        report = audit[audit["Status"] != "Accepted"]

    if report.empty:
        print("No matching submissions or problems found.")
    else:
        display(report[["Row", "Team", "Attempt", "Status", "Detail"]])

    return report


In [24]:
# CHEQUEA LOS ERRORES DE SUBMISION

_ = check_portfolio_submissions()

Total submissions: 2
Accepted submissions: 1
Submissions with problems: 1


,Row,Team,Attempt,Status,Detail
1,16,Team 3,1,Invalid team code — not counted,Incorrect or missing team code.


## ACTUALIZA LEADERBOARD MANUALMENTE

In [18]:
refresh_and_publish()

display(attempt_history.tail(3))

No new submissions. Leaderboard unchanged.


,Rank,Team,Performance,Attempts,Change
0,1,Team 3,100.0,1/4,—


,Row,Team,Attempt,Status,Detail
12,14,,0,Unknown team,Invalid team name.
13,15,Team 3,1,Accepted,
14,16,Team 3,1,Invalid team code — not counted,Incorrect or missing team code.


In [16]:
display(attempt_history.tail(5))

,Row,Team,Attempt,Status,Detail
9,11,,0,Unknown team,Invalid team name.
10,12,,0,Unknown team,Invalid team name.
11,13,,0,Unknown team,Invalid team name.
12,14,,0,Unknown team,Invalid team name.
13,15,Team 3,0,Invalid team code — not counted,Incorrect or missing team code.


# CHEQUEA PERIODICAMENTE SI HAY SUBMISIONES Y ACTUALIZA

In [15]:

import time
from IPython.display import clear_output

# Remember the last version of the submissions
last_submissions = None

try:
    while True:

        # Read the private Google Sheet
        current_submissions = submissions_sheet.get_all_values()

        # Update only when submissions have changed
        if current_submissions != last_submissions:

            clear_output(wait=True)

            refresh_and_publish()

            last_submissions = current_submissions

            print("Automatic leaderboard update completed.")

        # Check again after 30 seconds
        time.sleep(30)

except KeyboardInterrupt:
    print("Automatic updates stopped.")


Received: 14 submissions | Ranked: 0 teams


,Rank,Team,Performance,Attempts,Change


Automatic leaderboard update completed.
Automatic updates stopped.


In [19]:

# Get Team 3's official secret code
official_code = team_codes["Team 3"]

# Read the latest submissions from Google Sheets
submissions = pd.DataFrame(submissions_sheet.get_all_records())

# Select Team 3's most recent submission
team_submission = submissions[
    submissions["Team name"].str.strip() == "Team 3"
].iloc[-1]

# Get the submitted secret code
submitted_code = str(team_submission["Team Secret Code"])

# Compare both codes
print("Official code: ", repr(official_code))
print("Submitted code:", repr(submitted_code))

print("Codes match:",
      official_code.strip().upper() == submitted_code.strip().upper())


Official code:  'BQLWB2LE'
Submitted code: 'BQLWB2LE'
Codes match: True
